# Notebook 072: Star Schema Implementation: Fact and Conformed Dimension Modeling in Spark

**Module:** Module 07: Lakehouse Data Modeling, CDC & SCD Implementation  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 28: Data Modeling E2E Course & Video 26: Retail Data Engineering Project  

---

## 1. Business Scenario & Real-World Context
A retail enterprise transitions from a monolithic relational data warehouse to a Databricks Lakehouse. BI analysts need classic Star Schema structures (Sales Fact table surrounded by Date, Customer, Store, and Product conformed dimensions).

## 2. Technical Requirement & Problem Statement
Design and build a complete Star Schema in PySpark: construct conformed dimensions with surrogate keys, create the centralized Sales Fact table with foreign keys and degenerate dimensions, and execute star-join queries optimized for BI tools.

## 3. Spark & Databricks Architecture Notes
Star Schemas remain the gold standard for BI reporting in Lakehouses due to user familiarity and tool compatibility (Power BI / Tableau). Spark optimizes star joins using Dynamic Partition Pruning (DPP) and Broadcast Hash Joins.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_072_Star_Schema_Implementation_Fact_and_Conformed_Dimension_Modeling_in_Spark")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Building Conformed Dimensions (Customer & Product)
Construct dimension tables with surrogate keys.

In [ ]:
dim_cust = spark.createDataFrame([
    (101, "CUST-A", "Alice Smith", "Gold", "US"),
    (102, "CUST-B", "Bob Jones", "Silver", "UK")
], ["dim_customer_sk", "customer_id", "full_name", "loyalty_tier", "country"])

dim_prod = spark.createDataFrame([
    (501, "PROD-X", "Mechanical Keyboard", "Hardware", 89.99),
    (502, "PROD-Y", "USB-C Hub", "Hardware", 29.99)
], ["dim_product_sk", "product_id", "product_name", "category", "unit_price"])

print("Customer Dimension:")
dim_cust.show()
print("Product Dimension:")
dim_prod.show()

## 6. Building the Centralized Sales Fact Table
Construct fact table referencing dimension surrogate keys.

In [ ]:
fact_sales = spark.createDataFrame([
    ("TXN-901", 101, 501, 2, 179.98, "2023-11-20"),
    ("TXN-902", 101, 502, 1, 29.99,  "2023-11-20"),
    ("TXN-903", 102, 501, 1, 89.99,  "2023-11-20")
], ["txn_id", "dim_customer_sk", "dim_product_sk", "quantity", "total_sales_amount", "order_date"])

print("Sales Fact Table:")
fact_sales.show()

## 7. Executing Star-Join Query with Dynamic Pruning
Join fact table with dimensions for executive slice-and-dice reporting.

In [ ]:
df_star_report = (
    fact_sales
    .join(dim_cust, on="dim_customer_sk")
    .join(dim_prod, on="dim_product_sk")
    .groupBy("country", "category")
    .agg(
        F.sum("total_sales_amount").alias("gross_revenue"),
        F.sum("quantity").alias("units_sold")
    )
    .orderBy(F.desc("gross_revenue"))
)

print("Star-Schema Analytics Report:")
df_star_report.show()

## 9. Key Architectural Takeaways
- **Star Schemas feature a central numeric Fact table surrounded by denormalized Dimension tables.**
- **Surrogate keys (integers) improve join performance and decouple reporting from source transactional IDs.**
- **Dynamic Partition Pruning (DPP) allows Spark to skip fact table partitions dynamically based on dimension filters.**

## 10. Senior Data Engineering Interview Questions
1. How does Dynamic Partition Pruning (DPP) optimize Star Schema queries in Apache Spark?
2. What is the difference between a Star Schema and a Snowflake Schema in terms of normalization and query performance?
3. What is a Degenerate Dimension and where is it stored in a Star Schema?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
